# Model - 1

In [ ]:
import os
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image
from PIL import Image
import pandas as pd

class TwoClassImageDataset(Dataset):
    def __init__(self, mug_dir, bwmug_dir, transform=None):
        self.image_paths = []
        self.labels = []
        self.transform = transform
        image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.avif']

        for ext in image_extensions:
            self.image_paths += glob.glob(os.path.join(mug_dir, ext))
            self.labels += [0] * len(glob.glob(os.path.join(mug_dir, ext)))
            self.image_paths += glob.glob(os.path.join(bwmug_dir, ext))
            self.labels += [1] * len(glob.glob(os.path.join(bwmug_dir, ext)))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        label = self.labels[idx]
        try:
            with Image.open(path) as img:
                if img.mode == 'P':
                    img = img.convert('RGBA')
                image = img.convert('RGB')
        except Exception:
            return self.__getitem__((idx + 1) % len(self))
        if self.transform:
            image = self.transform(image)
        return image, label

class Generator(nn.Module):
    def __init__(self, nz, nc, ngf, num_classes, emb_size=100):
        super(Generator, self).__init__()
        self.label_emb = nn.Embedding(num_classes, emb_size)
        self.model = nn.Sequential(
            nn.ConvTranspose2d(nz + emb_size, ngf * 4, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),

            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),

            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),

            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, noise, labels):
        label_embedding = self.label_emb(labels).unsqueeze(2).unsqueeze(3)
        input = torch.cat((noise, label_embedding), 1)
        return self.model(input)

class Discriminator(nn.Module):
    def __init__(self, nc, ndf, num_classes, emb_size=100):
        super(Discriminator, self).__init__()
        self.label_emb = nn.Embedding(num_classes, emb_size)
        self.model = nn.Sequential(
            nn.Conv2d(nc + emb_size, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf * 2, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, img, labels):
        label_embedding = self.label_emb(labels).unsqueeze(2).unsqueeze(3)
        label_embedding = label_embedding.expand(labels.size(0), -1, img.size(2), img.size(3))
        input = torch.cat((img, label_embedding), 1)
        return self.model(input).mean([2, 3]).view(-1)

def weights_init(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.BatchNorm2d)):
        nn.init.normal_(m.weight.data, 0.0, 0.02)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
image_size = 28
nz = 100
ngf = 32
ndf = 32
nc = 3
num_classes = 2
lr = 0.0002
beta1 = 0.5
num_epochs = 500
batch_size = 64

base_path = "/content/drive/MyDrive/Team Project"
mug_dir = os.path.join(base_path, "mug")
bwmug_dir = os.path.join(base_path, "bwmug")
save_dir = base_path

transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
dataset = TwoClassImageDataset(mug_dir, bwmug_dir, transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True, pin_memory=False, num_workers=0)

netG = Generator(nz, nc, ngf, num_classes).to(device)
netD = Discriminator(nc, ndf, num_classes).to(device)
netG.apply(weights_init)
netD.apply(weights_init)

criterion = nn.BCELoss()
optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, 0.999))
schedulerD = StepLR(optimizerD, step_size=100, gamma=0.1)
schedulerG = StepLR(optimizerG, step_size=100, gamma=0.1)

start_epoch = 0

fixed_noise = torch.randn(8, nz, 1, 1, device=device)
fixed_labels = torch.tensor([0, 1, 0, 1, 0, 1, 0, 1], device=device)
history = []

print("Training started...")
for epoch in range(start_epoch, num_epochs):
    print(f"\nEpoch {epoch+1} started...")
    lossD_epoch = 0.0
    lossG_epoch = 0.0

    for i, (real_imgs, labels) in enumerate(dataloader):
        b_size = real_imgs.size(0)
        real_imgs, labels = real_imgs.to(device), labels.to(device)
        real_label = torch.full((b_size,), 1., device=device)
        fake_label = torch.full((b_size,), 0., device=device)

        netD.zero_grad()
        output_real = netD(real_imgs, labels)
        loss_real = criterion(output_real, real_label)

        noise = torch.randn(b_size, nz, 1, 1, device=device)
        fake_imgs = netG(noise, labels)
        output_fake = netD(fake_imgs.detach(), labels)
        loss_fake = criterion(output_fake, fake_label)

        lossD = loss_real + loss_fake
        lossD.backward()
        optimizerD.step()

        netG.zero_grad()
        output = netD(fake_imgs, labels)
        lossG = criterion(output, real_label)
        lossG.backward()
        optimizerG.step()

        lossD_epoch += lossD.item()
        lossG_epoch += lossG.item()

    if (epoch + 1) % 10 == 0:
        with torch.no_grad():
            fake = netG(fixed_noise, fixed_labels).detach().cpu()
            filename = f"cgan_epoch_{epoch+1}.png"
            save_image(fake, os.path.join(save_dir, filename), nrow=4, normalize=True)
            print(f"Saved: {filename}")

    if (epoch + 1) % 10 == 0:
        torch.save({
            'epoch': epoch + 1,
            'netG_state_dict': netG.state_dict(),
            'netD_state_dict': netD.state_dict(),
            'optimizerG_state_dict': optimizerG.state_dict(),
            'optimizerD_state_dict': optimizerD.state_dict(),
        }, os.path.join(save_dir, f"cgan_checkpoint_epoch_{epoch+1}.pth"))

    schedulerD.step()
    schedulerG.step()

    avg_lossD = lossD_epoch / len(dataloader)
    avg_lossG = lossG_epoch / len(dataloader)
    history.append([epoch + 1, avg_lossD, avg_lossG])
    print(f"Epoch {epoch+1} done. Loss_D: {avg_lossD:.4f}, Loss_G: {avg_lossG:.4f}")

# Model - 2

In [ ]:
import os, glob
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import make_grid, save_image

device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size  = 128
image_size  = 64
nz          = 100
num_classes = 2
ngf, ndf    = 64, 64
lr          = 2e-4
beta1       = 0.5
epochs      = 100

base_drive  = '/content/gdrive/MyDrive/Team Project'
results_dir = os.path.join(base_drive, 'CGAN_Results')
os.makedirs(results_dir, exist_ok=True)
log_path = os.path.join(results_dir, 'training_log.csv')
with open(log_path, 'w') as f:
    f.write('epoch,loss_d,loss_g\n')

class CupMugDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.samples = []
        for label, sub in enumerate(['cup','mug']):
            folder = os.path.join(root_dir, sub)
            for ext in ('jpg','jpeg','png','bmp','tiff','webp'):
                self.samples += [(p,label) for p in glob.glob(os.path.join(folder,f'*.{ext}'))]
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label

transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5,)*3, (0.5,)*3),
])
dataset = CupMugDataset(base_drive, transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

class SelfAttention(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.query = nn.Conv2d(in_dim, in_dim//8, 1)
        self.key   = nn.Conv2d(in_dim, in_dim//8, 1)
        self.value = nn.Conv2d(in_dim, in_dim,    1)
        self.gamma = nn.Parameter(torch.zeros(1))
        self.softmax = nn.Softmax(dim=-1)
    def forward(self, x):
        B,C,W,H = x.size()
        q = self.query(x).view(B,-1,W*H).permute(0,2,1)
        k = self.key(x).view(B,-1,W*H)
        att = self.softmax(torch.bmm(q,k))
        v = self.value(x).view(B,-1,W*H)
        o = torch.bmm(v, att.permute(0,2,1)).view(B,C,W,H)
        return self.gamma*o + x

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, num_classes)
        self.init = nn.Sequential(
            nn.ConvTranspose2d(nz+num_classes, ngf*8, 4,1,0,bias=False),
            nn.BatchNorm2d(ngf*8), nn.ReLU(True),
        )
        self.up1  = nn.Sequential(
            nn.ConvTranspose2d(ngf*8, ngf*4, 4,2,1,bias=False),
            nn.BatchNorm2d(ngf*4), nn.ReLU(True),
        )
        self.attn = SelfAttention(ngf*4)
        self.up2  = nn.Sequential(
            nn.ConvTranspose2d(ngf*4, ngf*2, 4,2,1,bias=False),
            nn.BatchNorm2d(ngf*2), nn.ReLU(True),
        )
        self.up3  = nn.Sequential(
            nn.ConvTranspose2d(ngf*2, ngf,   4,2,1,bias=False),
            nn.BatchNorm2d(ngf),   nn.ReLU(True),
        )
        self.final= nn.ConvTranspose2d(ngf,3,4,2,1,bias=False)
        self.tanh = nn.Tanh()
    def forward(self, noise, labels):
        x = torch.cat([noise, self.label_emb(labels)], dim=1).unsqueeze(2).unsqueeze(3)
        x = self.init(x); x = self.up1(x)
        x = self.attn(x); x = self.up2(x)
        x = self.up3(x); return self.tanh(self.final(x))

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, num_classes)
        self.init = nn.Sequential(
            nn.Conv2d(3+num_classes, ndf, 4,2,1,bias=False),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.down1 = nn.Sequential(
            nn.Conv2d(ndf, ndf*2,4,2,1,bias=False),
            nn.BatchNorm2d(ndf*2), nn.LeakyReLU(0.2, inplace=True),
        )
        self.attn  = SelfAttention(ndf*2)
        self.down2 = nn.Sequential(
            nn.Conv2d(ndf*2, ndf*4,4,2,1,bias=False),
            nn.BatchNorm2d(ndf*4), nn.LeakyReLU(0.2, inplace=True),
        )
        self.final = nn.Conv2d(ndf*4,1,4,1,0,bias=False)
        self.sig   = nn.Sigmoid()
    def forward(self, imgs, labels):
        lbl = self.label_emb(labels).unsqueeze(2).unsqueeze(3)
        lbl_map = lbl.repeat(1,1,image_size,image_size)
        x = torch.cat([imgs, lbl_map], dim=1)
        x = self.init(x); x = self.down1(x)
        x = self.attn(x); x = self.down2(x)
        out = self.final(x)           # (B,1,H',W')
        out = self.sig(out)
        # flatten per-sample and average:
        return out.view(out.size(0), -1).mean(1)

netG = Generator().to(device)
netD = Discriminator().to(device)
criterion = nn.BCELoss()
optG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1,0.999))
optD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1,0.999))

fixed_noise  = torch.randn(16, nz, device=device)
fixed_labels = torch.tensor([0]*8 + [1]*8, device=device)

for ep in range(1, epochs+1):
    ld_acc, lg_acc = 0.0, 0.0
    for imgs, labels in dataloader:
        imgs, labels = imgs.to(device), labels.to(device)
        b = imgs.size(0)
        real = torch.ones(b, device=device)
        fake = torch.zeros(b, device=device)

        netD.zero_grad()
        out_r = netD(imgs, labels)
        l_dr = criterion(out_r, real)
        l_dr.backward()

        noise = torch.randn(b, nz, device=device)
        rl    = torch.randint(0, num_classes, (b,), device=device)
        gen   = netG(noise, rl)
        out_f = netD(gen.detach(), rl)
        l_df  = criterion(out_f, fake)
        l_df.backward()
        optD.step()

        netG.zero_grad()
        out_g = netD(gen, rl)
        l_g   = criterion(out_g, real)
        l_g.backward()
        optG.step()

        ld_acc += (l_dr + l_df).item()
        lg_acc += l_g.item()

    ld = ld_acc / len(dataloader)
    lg = lg_acc / len(dataloader)
    print(f"[{ep}/{epochs}] Loss_D: {ld:.4f}  Loss_G: {lg:.4f}")
    with open(log_path,'a') as f:
        f.write(f"{ep},{ld:.4f},{lg:.4f}\n")

    if ep % 10 == 0:
        with torch.no_grad():
            samp = netG(fixed_noise, fixed_labels)
            grid = make_grid(samp, nrow=4, normalize=True, value_range=(-1,1))
            save_image(grid, os.path.join(results_dir, f"cgan_ep{ep}.png"))

# Model - 3

In [ ]:
import os, glob, csv
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image
from PIL import Image
import lpips

class TwoClassImageDataset(Dataset):
    def __init__(self, colored_dir, bw_dir, transform=None):
        self.image_paths, self.labels = [], []
        self.transform = transform

        for path in glob.glob(f"{colored_dir}/*"):
            self.image_paths.append(path)
            self.labels.append(0)

        for path in glob.glob(f"{bw_dir}/*"):
            self.image_paths.append(path)
            self.labels.append(1)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return img, label

class Generator(nn.Module):
    def __init__(self, nz, ngf=64, nc=3, num_classes=2):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, num_classes)
        self.nz = nz + num_classes
        self.main = nn.Sequential(
            nn.ConvTranspose2d(self.nz, ngf*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf*8),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf*8, ngf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*4),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*2),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf*2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, noise, labels):
        label_embedding = self.label_emb(labels)
        input = torch.cat([noise, label_embedding], 1).view(noise.size(0), -1, 1, 1)
        return self.main(input)

class Discriminator(nn.Module):
    def __init__(self, nc=3, ndf=64, num_classes=2):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, num_classes)

        self.main = nn.Sequential(
            nn.Conv2d(nc + num_classes, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*4, 1, 8, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x, labels):
        label_embed = self.label_emb(labels).unsqueeze(2).unsqueeze(3)
        label_embed = label_embed.expand(x.size(0), -1, x.size(2), x.size(3))
        x = torch.cat([x, label_embed], 1)
        out = self.main(x)
        return out.view(-1)

colored_dir = '/content/drive/MyDrive/Team_Project/Team Project/cup'
bw_dir = '/content/drive/MyDrive/Team_Project/Team Project/bw cup'
save_dir = '/content/drive/MyDrive/CGAN/Cup/3.0generated_images'
csv_path = os.path.join(save_dir, 'training_log_lpips.csv')
os.makedirs(save_dir, exist_ok=True)

nz, image_size = 100, 64
batch_size = 64
num_epochs = 200
lr, beta1, beta2 = 0.0002, 0.5, 0.999
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])
dataset = TwoClassImageDataset(colored_dir, bw_dir, transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

netG = Generator(nz=nz).to(device)
netD = Discriminator().to(device)
criterion = nn.BCELoss()
perceptual_loss = lpips.LPIPS(net='alex').to(device)
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, beta2))
optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, beta2))

fixed_noise = torch.randn(25, nz, device=device)
fixed_labels = torch.cat([torch.zeros(13, dtype=torch.long), torch.ones(12, dtype=torch.long)]).to(device)

print("Eğitim başlıyor...")

for epoch in range(num_epochs):
    print(f"{epoch + 1}. epoch başladı")
    lossD_total, lossG_total = 0, 0
    for real_imgs, labels in dataloader:
        real_imgs, labels = real_imgs.to(device), labels.to(device)
        b_size = real_imgs.size(0)

        netD.zero_grad()
        real_labels = torch.full((b_size,), 1., dtype=torch.float, device=device)
        fake_labels = torch.full((b_size,), 0., dtype=torch.float, device=device)

        output_real = netD(real_imgs, labels)
        lossD_real = criterion(output_real, real_labels)

        noise = torch.randn(b_size, nz, device=device)
        fake_imgs = netG(noise, labels)
        output_fake = netD(fake_imgs.detach(), labels)
        lossD_fake = criterion(output_fake, fake_labels)

        lossD = lossD_real + lossD_fake
        lossD.backward()
        optimizerD.step()

        netG.zero_grad()
        output = netD(fake_imgs, labels)
        gan_loss = criterion(output, real_labels)
        lpips_val = perceptual_loss(fake_imgs, real_imgs).mean()
        lossG = gan_loss + lpips_val
        lossG.backward()
        optimizerG.step()

        lossD_total += lossD.item()
        lossG_total += lossG.item()

    with open(csv_path, 'a', newline='') as f:
        writer = csv.writer(f)
        if epoch == 0:
            writer.writerow(['Epoch', 'Loss_D', 'Loss_G'])
        writer.writerow([epoch+1, lossD_total, lossG_total])

    if (epoch + 1) % 10 == 0:
        with torch.no_grad():
            gen_imgs = netG(fixed_noise, fixed_labels).detach().cpu()
        save_image(gen_imgs, os.path.join(save_dir, f'generated_epoch_{epoch+1}.jpg'), nrow=5, normalize=True)

        torch.save({
            'epoch': epoch + 1,
            'netG_state_dict': netG.state_dict(),
            'netD_state_dict': netD.state_dict(),
            'optimizerG_state_dict': optimizerG.state_dict(),
            'optimizerD_state_dict': optimizerD.state_dict(),
        }, os.path.join(save_dir, f'checkpoint_epoch_{epoch+1}.pt'))

        print(f"Epoch {epoch+1}: Görsel ve checkpoint kaydedildi.")

    print(f"{epoch+1}. epoch bitti | D: {lossD_total:.4f}, G: {lossG_total:.4f}, LPIPS: {lpips_val.item():.4f}")

# Model - 4 (Because of the outcome we destroyed it.)

# Model - 5

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image, make_grid
from PIL import Image
import torchvision

class TwoClassImageDataset(Dataset):
    def __init__(self, cup_dir, mug_dir, transform=None):
        self.cup_images = [(os.path.join(cup_dir, img), 0) for img in os.listdir(cup_dir)]
        self.mug_images = [(os.path.join(mug_dir, img), 1) for img in os.listdir(mug_dir)]
        self.all_images = self.cup_images + self.mug_images
        self.transform = transform

    def __len__(self):
        return len(self.all_images)

    def __getitem__(self, idx):
      path, label = self.all_images[idx]
      image = Image.open(path)
      if image.mode == 'P' or image.mode == 'RGBA':
          image = image.convert('RGBA').convert('RGB')
      else:
          image = image.convert('RGB')
      if self.transform:
          image = self.transform(image)
      return image, label

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

cup_dir = "/content/drive/MyDrive/Team_Project/Team Project/cup"
mug_dir = "/content/drive/MyDrive/Team_Project/Team Project/mug"

dataset = TwoClassImageDataset(cup_dir, mug_dir, transform=transform)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True, drop_last=True)

class Generator(nn.Module):
    def __init__(self, noise_dim, num_classes):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, num_classes)
        self.net = nn.Sequential(
            nn.Linear(noise_dim + num_classes, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(True),
            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(True),
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(True),
            nn.Linear(1024, 3 * 64 * 64),
            nn.Tanh()
        )

    def forward(self, noise, labels):
        labels_onehot = self.label_emb(labels)
        x = torch.cat([noise, labels_onehot], dim=1)
        out = self.net(x)
        return out.view(out.size(0), 3, 64, 64)

class Discriminator(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.label_embedding = nn.Embedding(num_classes, num_classes)
        self.net = nn.Sequential(
            nn.Linear(3 * 64 * 64 + num_classes, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, img, labels):
        x = img.view(img.size(0), -1)
        labels_onehot = self.label_embedding(labels)
        x = torch.cat([x, labels_onehot], dim=1)
        return self.net(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
noise_dim = 100
real_label = 0.9
fake_label = 0.0
start_epoch = 400
epochs = 1000
save_dir = "/content/drive/MyDrive/CGAN_checkpoints"
os.makedirs(save_dir, exist_ok=True)

G = Generator(noise_dim, 2).to(device)
D = Discriminator(2).to(device)
G.load_state_dict(torch.load(os.path.join(save_dir, f"G_epoch_{start_epoch}.pt")))
D.load_state_dict(torch.load(os.path.join(save_dir, f"D_epoch_{start_epoch}.pt")))

optimizer_G = optim.Adam(G.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D = optim.Adam(D.parameters(), lr=0.0002, betas=(0.5, 0.999))
criterion = nn.BCELoss()

for epoch in range(start_epoch, epochs):
    print(f"Epoch {epoch+1} started...")
    for i, (real_images, labels) in enumerate(dataloader):
        batch_size = real_images.size(0)
        real_images, labels = real_images.to(device), labels.to(device)

        D.zero_grad()
        outputs_real = D(real_images, labels).view(-1)
        loss_real = criterion(outputs_real, torch.full_like(outputs_real, real_label, device=device))

        noise = torch.randn(batch_size, noise_dim, device=device)
        fake_labels = torch.randint(0, 2, (batch_size,), device=device)
        fake_images = G(noise, fake_labels)
        outputs_fake = D(fake_images.detach(), fake_labels).view(-1)
        loss_fake = criterion(outputs_fake, torch.full_like(outputs_fake, fake_label, device=device))

        loss_D = loss_real + loss_fake
        loss_D.backward()
        optimizer_D.step()

        G.zero_grad()
        outputs = D(fake_images, fake_labels).view(-1)
        loss_G = criterion(outputs, torch.full_like(outputs, real_label, device=device))
        loss_G.backward()
        optimizer_G.step()

    print(f"Epoch {epoch+1} ended. Loss_D: {loss_D.item():.4f}, Loss_G: {loss_G.item():.4f}")

    if (epoch + 1) % 10 == 0:
        torch.save(G.state_dict(), os.path.join(save_dir, f"G_epoch_{epoch+1}.pt"))
        torch.save(D.state_dict(), os.path.join(save_dir, f"D_epoch_{epoch+1}.pt"))

        G.eval()
        with torch.no_grad():
            for label in [0, 1]:
                noise = torch.randn(8, noise_dim, device=device)
                labels_fixed = torch.full((8,), label, dtype=torch.long, device=device)
                fake_imgs = G(noise, labels_fixed)
                grid = make_grid(fake_imgs, nrow=4, normalize=True)
                save_path = os.path.join(save_dir, f"samples_epoch_{epoch+1}_label_{label}.png")
                save_image(grid, save_path)
        G.train()

# Model - 6

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import make_grid, save_image
from PIL import Image
import lpips

class TwoClassImageDataset(Dataset):
    def __init__(self, cup_dir, mug_dir, transform=None):
        self.cup_images = [(os.path.join(cup_dir, img), 0) for img in os.listdir(cup_dir)]
        self.mug_images = [(os.path.join(mug_dir, img), 1) for img in os.listdir(mug_dir)]
        self.all_images = self.cup_images + self.mug_images
        self.transform = transform

    def __len__(self):
        return len(self.all_images)

    def __getitem__(self, idx):
        path, label = self.all_images[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

class Generator(nn.Module):
    def __init__(self, noise_dim, num_classes):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, num_classes)
        self.net = nn.Sequential(
            nn.Linear(noise_dim + num_classes, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(True),
            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(True),
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(True),
            nn.Linear(1024, 3 * 64 * 64),
            nn.Tanh()
        )

    def forward(self, noise, labels):
        labels_onehot = self.label_emb(labels)
        x = torch.cat([noise, labels_onehot], dim=1)
        out = self.net(x)
        return out.view(out.size(0), 3, 64, 64)

class Discriminator(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.label_embedding = nn.Embedding(num_classes, num_classes)
        self.net = nn.Sequential(
            nn.Linear(3 * 64 * 64 + num_classes, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, img, labels):
        x = img.view(img.size(0), -1)
        labels_onehot = self.label_embedding(labels)
        x = torch.cat([x, labels_onehot], dim=1)
        return self.net(x)

save_dir = "/content/drive/MyDrive/CGAN_checkpoints/LPIPS/3.0"
os.makedirs(save_dir, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
noise_dim = 100
real_label = 0.9
fake_label = 0.0
epochs = 500
lpips_lambda = 0.1

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])
cup_dir = "/content/drive/MyDrive/Team_Project/Team Project/cup"
mug_dir = "/content/drive/MyDrive/Team_Project/Team Project/mug"
dataset = TwoClassImageDataset(cup_dir, mug_dir, transform=transform)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True, drop_last=True)

G = Generator(noise_dim, 2).to(device)
D = Discriminator(2).to(device)

criterion = nn.BCELoss()
optimizer_G = optim.Adam(G.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D = optim.Adam(D.parameters(), lr=0.0002, betas=(0.5, 0.999))

lpips_metric = lpips.LPIPS(net='alex').to(device).eval()

for epoch in range(epochs):
    print(f"Epoch {epoch+1} started...")
    lpips_total = 0.0
    lpips_batches = 0

    for i, (real_images, labels) in enumerate(dataloader):
        batch_size = real_images.size(0)
        real_images, labels = real_images.to(device), labels.to(device)

        D.zero_grad()
        output_real = D(real_images, labels).view(-1)
        loss_real = criterion(output_real, torch.full_like(output_real, real_label, device=device))

        noise = torch.randn(batch_size, noise_dim, device=device)
        fake_labels = torch.randint(0, 2, (batch_size,), device=device)
        fake_images = G(noise, fake_labels)
        output_fake = D(fake_images.detach(), fake_labels).view(-1)
        loss_fake = criterion(output_fake, torch.full_like(output_fake, fake_label, device=device))

        loss_D = loss_real + loss_fake
        loss_D.backward()
        optimizer_D.step()

        G.zero_grad()
        output = D(fake_images, fake_labels).view(-1)
        bce_loss = criterion(output, torch.full_like(output, real_label, device=device))

        fake_lp = (fake_images + 1) / 2 * 2 - 1
        real_lp = (real_images + 1) / 2 * 2 - 1
        lpips_loss = lpips_metric(fake_lp, real_lp).mean()

        loss_G = bce_loss + lpips_lambda * lpips_loss
        loss_G.backward()
        optimizer_G.step()

        if i % 5 == 0:
            with torch.no_grad():
                lpips_total += lpips_loss.item()
                lpips_batches += 1

    with torch.no_grad():
        G.eval()
        noise = torch.randn(64, noise_dim, device=device)
        fixed_labels = torch.randint(0, 2, (64,), device=device)
        fake_imgs = G(noise, fixed_labels)
        real_batch = next(iter(dataloader))[0].to(device)[:64]
        real_lp = (real_batch + 1) / 2 * 2 - 1
        fake_lp = (fake_imgs + 1) / 2 * 2 - 1
        epoch_lpips = lpips_metric(fake_lp, real_lp).mean().item()
        G.train()

    avg_lpips = lpips_total / lpips_batches if lpips_batches > 0 else 0
    print(f"Epoch {epoch+1} ended. Loss_D: {loss_D.item():.4f}, Loss_G: {loss_G.item():.4f}, Batch LPIPS: {avg_lpips:.4f}, Epoch LPIPS: {epoch_lpips:.4f}")

    if (epoch + 1) % 10 == 0:
        torch.save(G.state_dict(), os.path.join(save_dir, f"G_epoch_{epoch+1}.pt"))
        torch.save(D.state_dict(), os.path.join(save_dir, f"D_epoch_{epoch+1}.pt"))

        with torch.no_grad():
            for label in [0, 1]:
                test_noise = torch.randn(8, noise_dim, device=device)
                label_tensor = torch.full((8,), label, dtype=torch.long, device=device)
                fake_imgs = G(test_noise, label_tensor)
                grid = make_grid(fake_imgs, nrow=4, normalize=True)
                save_path = os.path.join(save_dir, f"samples_epoch_{epoch+1}_label_{label}.png")
                save_image(grid, save_path)


# Model - 7

In [ ]:
import os, glob
from PIL import Image
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import make_grid, save_image
from torch.cuda.amp import autocast, GradScaler

device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
image_size  = 32
batch_size  = 256
nz          = 100
num_classes = 2
lr          = 1e-4
beta1, beta2= 0.0, 0.9
n_critic    = 5
lambda_gp   = 10.0
epochs      = 1000

base_dir    = '/content/gdrive/MyDrive/Team Project'
out_dir     = os.path.join(base_dir, 'WGAN_GP_Results')
os.makedirs(out_dir, exist_ok=True)
log_path    = os.path.join(out_dir, 'log.csv')
ckpt_path   = os.path.join(out_dir, 'ckpt.pth')

if not os.path.isfile(log_path):
    with open(log_path,'w') as f:
        f.write('epoch,critic_loss,gen_loss\n')

class CupMug(Dataset):
    def __init__(self,root,tfm):
        self.samples=[]
        for lbl,sub in enumerate(['cup','mug']):
            folder = os.path.join(root,sub)
            for ext in ('jpg','png','jpeg','bmp'):
                self.samples += [(p,lbl) for p in glob.glob(f"{folder}/*.{ext}")]
        self.tfm = tfm
    def __len__(self): return len(self.samples)
    def __getitem__(self,i):
        p,lbl = self.samples[i]
        img = Image.open(p).convert('RGB')
        return self.tfm(img), lbl

tfm = transforms.Compose([
    transforms.Resize(image_size),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5,)*3,(0.5,)*3),
])

dataset   = CupMug(base_dir, tfm)
dataloader= DataLoader(dataset, batch_size=batch_size, shuffle=True,
                       num_workers=4, pin_memory=True)

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(num_classes, num_classes)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(nz+num_classes,256,4,1,0,bias=False),
            nn.BatchNorm2d(256), nn.ReLU(True),
            nn.ConvTranspose2d(256,128,4,2,1,bias=False),
            nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128,64,4,2,1,bias=False),
            nn.BatchNorm2d(64), nn.ReLU(True),
            nn.ConvTranspose2d(64,3,4,2,1,bias=False),
            nn.Tanh()
        )
    def forward(self,z,y):
        yv = self.embed(y).unsqueeze(-1).unsqueeze(-1)
        x  = torch.cat([z, yv.expand(-1,-1,1,1)],1)
        return self.net(x)

class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(num_classes, num_classes)
        self.net = nn.Sequential(
            nn.Conv2d(3+num_classes,64,4,2,1,bias=False),
            nn.LeakyReLU(0.2,True),
            nn.Conv2d(64,128,4,2,1,bias=False),
            nn.LayerNorm([128,8,8]), nn.LeakyReLU(0.2,True),
            nn.Conv2d(128,256,4,2,1,bias=False),
            nn.LayerNorm([256,4,4]), nn.LeakyReLU(0.2,True),
            nn.Conv2d(256,1,4,1,0,bias=False),
        )
    def forward(self,img,y):
        yv = self.embed(y).unsqueeze(-1).unsqueeze(-1)
        x  = torch.cat([img, yv.expand(-1,-1,image_size,image_size)],1)
        return self.net(x).view(-1)

def gradient_penalty(D, real, fake, labels):
    B = real.size(0)
    alpha = torch.rand(B,1,1,1,device=device)
    interp = alpha*real + (1-alpha)*fake
    interp.requires_grad_(True)
    d_interp = D(interp, labels)
    grad = torch.autograd.grad(
        outputs=d_interp, inputs=interp,
        grad_outputs=torch.ones_like(d_interp),
        create_graph=True, retain_graph=True
    )[0]
    gp = ((grad.view(B,-1).norm(2,dim=1)-1)**2).mean()
    return gp

G = Generator().to(device)
C = Critic().to(device)
optG = optim.Adam(G.parameters(), lr=lr, betas=(beta1,beta2))
optC = optim.Adam(C.parameters(), lr=lr, betas=(beta1,beta2))
scaler = GradScaler()

fixed_z = torch.randn(16, nz,1,1, device=device)
fixed_y = torch.tensor([0]*8+[1]*8, device=device)

start_ep = 210
if os.path.isfile(ckpt_path):
    ck = torch.load(ckpt_path, map_location=device)
    G.load_state_dict(ck['G'])
    C.load_state_dict(ck['C'])
    optG.load_state_dict(ck['optG'])
    optC.load_state_dict(ck['optC'])
    scaler.load_state_dict(ck['scaler'])
    print(f"Loaded checkpoint weights (forced to start at epoch {start_ep})")

for ep in range(start_ep, epochs+1):
    lc_avg, lg_avg = 0.0, 0.0
    for imgs, labs in dataloader:
        B = imgs.size(0)
        imgs, labs = imgs.to(device), labs.to(device)

        for _ in range(n_critic):
            z = torch.randn(B, nz,1,1, device=device)
            y = torch.randint(0, num_classes, (B,), device=device)
            with autocast():
                fake = G(z,y)
                real_score = C(imgs, labs)
                fake_score = C(fake.detach(), y)
                gp = gradient_penalty(C, imgs, fake.detach(), labs)
                lossC = fake_score.mean() - real_score.mean() + lambda_gp * gp
            scaler.scale(lossC).backward()
            scaler.step(optC); scaler.update()
            C.zero_grad()

        z = torch.randn(B, nz,1,1, device=device)
        y = torch.randint(0, num_classes, (B,), device=device)
        with autocast():
            fake = G(z,y)
            fake_score = C(fake, y)
            lossG = -fake_score.mean()
        scaler.scale(lossG).backward()
        scaler.step(optG); scaler.update()
        G.zero_grad()

        lc_avg += lossC.item()
        lg_avg += lossG.item()

    lc = lc_avg/len(dataloader)
    lg = lg_avg/len(dataloader)
    print(f"[{ep}/{epochs}] Critic: {lc:.4f}  Gen: {lg:.4f}")
    with open(log_path,'a') as f:
        f.write(f"{ep},{lc:.4f},{lg:.4f}\n")

    if ep % 10 == 0 or ep == epochs:
        with torch.no_grad():
            samp = G(fixed_z, fixed_y)
            grid = make_grid(samp, nrow=4, normalize=True, value_range=(-1,1))
            save_image(grid, os.path.join(out_dir, f"wgangp_ep{ep}.png"))
        torch.save({
            'epoch': ep, 'G':G.state_dict(), 'C':C.state_dict(),
            'optG':optG.state_dict(),'optC':optC.state_dict(),
            'scaler':scaler.state_dict()
        }, ckpt_path)
        print(f"--> Saved checkpoint @ epoch {ep}")